# Лабораторная работа 4. Введение в машинное обучение: knn, деревянные алгоритмы и ансамблирование. Реализация.

![](https://newapplift-production.s3.amazonaws.com/comfy/cms/files/files/000/001/201/original/machine-learning-robots-dilbert.gif)

Результат лабораторной работы − отчет. Мы предпочитаем принимать отчеты в формате ноутбуков Jupyter (ipynb-файл). Постарайтесь сделать ваш отчет интересным рассказом, последовательно отвечающим на вопросы из заданий. Помимо ответов на вопросы, в отчете так же должен быть код, однако чем меньше кода, тем лучше всем: нам − меньше проверять, вам —  проще найти ошибку или дополнить эксперимент. При проверке оценивается четкость ответов на вопросы, аккуратность отчета и кода.


### Оценивание и штрафы
* Не копируйте классы между заданиями, объявите решающие модели один раз, а затем их инстанциируйте в каждой из ячеек
* Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи).
* Максимально допустимая оценка за работу — 14 балла. Дополнительных баллов не предусмотрено.
* Сдавать задание после указанного срока сдачи нельзя.
* Не оцениваются задания с удалёнными формулировками.
* Не оценивается лабораторная работа целиком, если она была выложена в открытый источник.


## Метрика качества

Обучение и оценка качества модели производятся на независимых множествах примеров. Как правило, имеющующиеся примеры разбивают на два подмножества: обучающее (train) и тестовое (test). Выбор пропорции разбиения — компромисс. Действительно, большой размер обучения ведет к более качественным алгоритмам, но бОльшему шуму при оценке модели на тесте. И наоборот, большой размер тестовой выборки ведет к менее шумной оценке качества, однако обученные модели получаются менее точными.

Многие модели классификации предсказывают оценку принадлежности положительному классу $\tilde{y}(x) \in R$ (например, вероятность принадлежности классу 1). После этого принимают решение о классе объекта путем сравнения оценки с некоторым порогом $\theta$:

$$y(x) = 
\begin{cases}
+1, &\text{если} \; \tilde{y}(x) \geq \theta \\
-1, &\text{если} \; \tilde{y}(x) < \theta
\end{cases}
$$

В этом случае можно рассматривать метрики, которые умеют работать с исходным ответом классификатора. В задании мы будем работать с метрикой AUC-ROC, которую в данном случае можно считать как долю неправильно упорядоченных пар объектов, отсортированных по возрастанию предсказанной оценки принадлежности классу 1 (более подробно можно узнать на следующих лекциях или, например, [здесь](https://github.com/esokolov/ml-course-msu/blob/master/ML15/lecture-notes/Sem05_metrics.pdf)). Детального понимания принципов работы метрики AUC-ROC для выполнения этой лабораторной не требуется. В sklearn данная метрика реализуется [следующей функцией](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).

# Реализации алгоритмов классификации (14 баллов)

В данном задании вам предстоит сделать реализации уже пройденных алгоритмов. Критерии, по которым задания будут засчитываться следующие:

1. Правильность работы алгоритмов стоит сравнивать на [датасете с ирисами](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html), если не сказано обратного.
2. Алгоритм должен работать не хуже [бейзлайнов](https://datascience.stackexchange.com/questions/30912/what-does-baseline-mean-in-the-context-of-machine-learning) из sklearn при одинаковых гиперпараметрах - допустима ошибка 20% относительно бейзлайнов из sklearn. Т.е. если ваш алгоритм имеет `AUC-ROC 0.4`, а бейзлайн из sklearn `AUC-ROC 0.9`, то допустимым скором будет $0.9 - 0.2 \cdot 0.9 = 0.72$, ваш же скор меньше допустимого и равен $0.4$ - такой алгоритм зачесть нельзя. А если у вас `AUC-ROC 0.85`, то ваш скор выше допустимого - такой алгоритм зачесть можно. Пороги ошибки могут увеличиться в пользу студентов.
3. Код должен быть написан аккуратно, с понятными именами, разделением на функции и так далее. Чем выше неаккуратность кода, тем больше вероятность, что ~~его читать не будут~~ вы не получите комметариев, если вдруг у вас что-то не заведётся.
4. Никто не запрещает вам разбираться в готовых реализациях из интернета, мы даже это поощряем, не поощряем только бездумной копипасты. Если код будет содержать дополнительную функциональность, о которой не сказано в задании и вы не объясните, зачем вы её использовали и почему именно её - вам будет выставлено 0 баллов за весь раздел. Т.е. если вы ~~скопировали~~ реализовали decision tree и там используется специальная регуляризации в листьях дерева и вы не объясните зачем она нужна и почему она там, то вам за весь раздел с decision tree будет выставлено 0 баллов.

In [2]:
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True, as_frame=True)

## decision tree (13 баллов)

Интерфейс можете подсмотреть в [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier)

В данном разделе вам предстоит реализовать decision tree алгоритм для задачи классификации. Он должен поддерживать следующие параметры:
- максимальная глубина дерева (*max_depth*)



### plain decision tree (5.5 баллов)

Реализовать алгоритм решающего дерева, продолжив код с семинара. Теорию можно найти в ученике в главе [Решающие деревья](https://education.yandex.ru/handbook/ml/article/reshayushchiye-derevya)


**Задание 1. Наивная реализация decision tree (2б)**

Реализовать decision tree алгоритм с фиксированной глубиной дерева h=6. Если у вас алгоритм работает дольше, можете уменьшить глубину до 5 или 4.

В качестве алгоритма взять `Жадный алгоритм построения решающего дерева`. В качестве критерия ветвления выберите misclassification error. В данном задании вам не нужно делать никаких оптимизаций, ожидается наивное решение со сложностью $\mathcal{O}(hN^2D)$, где $h$ - высота дерева, $N$ - количество обучающих семплов и $D$ - размерность пространства фичей.


In [6]:
# TODO: code here

**Задание 2. Модификация алгоритма для предсказания вероятностей классов (1б)**

Поддержите критейри Джини для построения дерева. Сравните результаты работы этих двух критерием на задаче классификации iris.


In [7]:
# TODO: code here

**Задание 3. Использование динамического программирования для ускорения алгоритма (2.5б)**

Используйте динамическое программирование, которое описано в главе Динамическое программирование учебника. Ваша сложность должна быть в этом месте должна быть $\mathcal{O}(hND\log(N))$. Сравните результаты работы наивной реализации дерева решений с оптимизированной, покажите, как растёт скорость работы, замерьте качество двух моделей.

In [8]:
# TODO: code here


### enhanced decision tree (4.5 балла)

Для тестирования решение понадобится датасет с категориальными признаками. Датасет с описанием расположен [здесь](http://archive.ics.uci.edu/dataset/45/heart+disease).

Код рассчитан на то, что вы их распакуете в директорию `datasets` в рабочей директории вашего ноутбука.

**Задание 4. Подготовка. (0.5 балла)**

Скачайте датасет, выкините или обработайте в нём категориальные признаки и заполните пропуски константой. Запустите вашу реализацию из секции `plain decision tree` и замерьте ваш результат классификации.

In [107]:
import numpy as np
import pandas as pd

names=[
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

num_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
cat_feaures = ["exang", "restecg", "fbs", "cp", "sex", "thal", "slope", "ca"]

df_clev = pd.read_csv("datasets/processed.cleveland.data", header=None, names=names)
df_hung = pd.read_csv("datasets/processed.hungarian.data", header=None, names=names)
df_swit = pd.read_csv("datasets/processed.switzerland.data", header=None, names=names)
df_va   = pd.read_csv("datasets/processed.va.data", header=None, names=names)

heart_disease = pd.concat([
    df_clev.replace(to_replace="?", value=np.nan).astype(float),
    df_hung.replace(to_replace="?", value=np.nan).astype(float),
    df_swit.replace(to_replace="?", value=np.nan).astype(float),
    df_va.replace(to_replace="?", value=np.nan).astype(float),
])

X, y = heart_disease.drop(["target"], axis=1), (heart_disease["target"] > 0).astype(float)

Мы бинаризировали наши метки, чтобы решать задачу бинарной классификации:

In [108]:
y.value_counts()

1.0    509
0.0    411
Name: target, dtype: int64

В данных очень много пропусков:

In [105]:
print("Процент пропусков в числовых фичах: ")
(X[num_features].isna().sum(axis=0) / X.shape[0] * 100)

Процент пропусков в числовых фичах: 


age         0.000000
trestbps    6.413043
chol        3.260870
thalach     5.978261
oldpeak     6.739130
dtype: float64

In [106]:
print("Процент пропусков в категориальных фичах: ")
(X[cat_feaures].isna().sum(axis=0) / X.shape[0] * 100)

Процент пропусков в категориальных фичах: 


exang       5.978261
restecg     0.217391
fbs         9.782609
cp          0.000000
sex         0.000000
thal       52.826087
slope      33.586957
ca         66.413043
dtype: float64

In [109]:
# TODO: cat features and missing values default handling here

**Задание 5. Поддержка категориальных признаков (2 балла)**

Поддержите категориальные фичи, как это описано в параграфе учебника про особенности данных. Сами категориальные фичи вы можете посмотреть в описании датасетта на сайте.

Сравните результаты работы реализации из секции `plain decision tree` без явной поддержки категориальных фичей, как вы делали при подготовке, с поддержкой категориальных. Покажите, как растёт или падает качество обученной модели на отложенной выборке.

In [11]:
# TODO: code here

**Задание 6. Работа с пропусками (2 балла)**

Обработайте пропуски, как это описано в параграфе учебника про особенности данных.

Сравните результаты работы реализации из секции `plain decision tree` с заполнением пропусков, как вы делали при подготовке, с поддержкой пропусков в фичах. Покажите, как растёт или падает качество обученной модели на отложенной выборке.

In [12]:
# TODO: code here

## random forest (4 балла)

Теорию для данного задания вы можете найти в соответсвующей [главе учебника](https://education.yandex.ru/handbook/ml/article/ansambli-v-mashinnom-obuchenii).

В данной разделе вы можете использовать как вашу реализацию из предыдущего раздела, так и реализацию дерева решений из sklearn. При использовании реализации sklearn вы получите не более половины баллов за задание, когда использование собственного дерева решений может вам принести полный балл.

Интерфейс можете подсмотреть в [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

random forest classifier должен поддерживать следующие параметры:
- максимальное количество деревьев в лесе (*n_estimators*)
- максимальное количество семплов для обучения одного дерева (*max_samples*)
- максимальная глубина дерева (*max_depth*)


**Задание 7. Беггинг над решающими деревьями (1 балла)**

Используйте беггинг над решающими деревьями. Помните, что беггинг призван уменьшить variance, но не bias. Чтобы ваше смещение было небольшим, берите достаточно глубокие деревья.

Сравните результаты работы реализации из секции `decision tree` с беггингом деревьев решений. Переберите размер выборки при бутстрапе, при котором у вашего беггинга будет максимальный скор. Удалось ли добиться роста метрики относительно базового алгоритма? Почему?

In [ ]:
# TODO: code here

**Задание 8. Feature subsampling (3 балла)**

Во время беггинга вводится предположение, что базовые алгоритмы некоррелированы. На самом деле это не так, потому что это одна модель, обучающаяся на данных из одного распределения. Чтобы разломать корреляцию используют семплирование фичей при обучении нового алгоритма: выбирают случайное подмножество фичей, на котором будет обучаться очередная модель.

Используйте feature subsampling во время беггинга:
1. фиксируйте набор фичей для построения дерева перед началом построения (1б)
2. выбирайте случайно множество фичей при выборе разделяющего правила во внутренней ноде (1б)

Сравните результаты работы реализации из секции `decision tree` c деревьями решений (1б). В качестве количества семплированных фичей возьмите квадратный корень из их общего количества, как было предложено в учебнике. Например, если у вас 25 исходных фичей, вы будете выбирать по 5 фичей для обучения очередной модели при беггинге. Переберите размер выборки при бутстрапе, при котором у вашего беггинга будет максимальный скор. Удалось ли добиться роста метрики относительно базового алгоритма и беггинга без feature subsampling? Почему?

In [15]:
# TODO: code here